In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
gdf = gpd.read_file('311_Encampment_Reports%2C_City_of_San_Diego%2C_2018.geojson')

In [ ]:
gdf.head()

In [ ]:
missing_count = gdf['public_des'].isnull().sum()

print(f"Number of missing records in public_des: {missing_count}")

In [ ]:
docs = gdf.public_des.tolist()
docs

In [ ]:
# Initialize empty lists to store cleaned docs and removed indices
cleaned_docs = []
removed_indices = []

# Loop through the list and keep track of indices
for idx, doc in enumerate(docs):
    if doc.strip() == '':
        removed_indices.append(idx)  # Track removed indices
    else:
        cleaned_docs.append(doc)  # Add non-blank items to cleaned list

print("Cleaned Docs:", cleaned_docs)
print("Removed Indices:", removed_indices)
print("Number of removed indices:", len(removed_indices))

In [ ]:
# Python program to generate WordCloud

# importing all necessary modules
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt
import pandas as pd


comment_words = ''
stopwords = set(STOPWORDS)

# iterate through the csv file
for val in cleaned_docs:
	
	# typecaste each val to string
	val = str(val)

	# split the value
	tokens = val.split()
	
	# Converts each token into lowercase
	for i in range(len(tokens)):
		tokens[i] = tokens[i].lower()
	
	comment_words += " ".join(tokens)+" "

wordcloud = WordCloud(width = 800, height = 800,
				background_color ='white',
				stopwords = stopwords,
				min_font_size = 10).generate(comment_words)

# plot the WordCloud image					 
plt.figure(figsize = (8, 8), facecolor = None)
plt.imshow(wordcloud)
plt.axis("off")
plt.tight_layout(pad = 0)

plt.show()


In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.corpus import stopwords

# If using sklearn stopwords
stop_words = list(ENGLISH_STOP_WORDS)

# Initialize empty lists to store cleaned docs and removed indices
cleaned_docs = []
removed_indices = []

# Loop through the list and keep track of indices
for idx, doc in enumerate(docs):
    # Remove stopwords
    cleaned_doc = ' '.join([word for word in doc.split() if word.lower() not in stop_words])
    
    # Check if the cleaned document is not empty
    if cleaned_doc.strip() == '':
        removed_indices.append(idx)  # Track removed indices
    else:
        cleaned_docs.append(cleaned_doc)  # Add non-blank, cleaned items to cleaned_docs

print("Cleaned Docs:", cleaned_docs)
print("Removed Indices:", removed_indices)
print("Number of removed indices:", len(removed_indices))

In [ ]:
cleaned_docs

In [ ]:
# Python program to generate WordCloud

# importing all necessary modules
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt
import pandas as pd


comment_words = ''
stopwords = set(STOPWORDS)

# iterate through the csv file
for val in cleaned_docs:
	
	# typecaste each val to string
	val = str(val)

	# split the value
	tokens = val.split()
	
	# Converts each token into lowercase
	for i in range(len(tokens)):
		tokens[i] = tokens[i].lower()
	
	comment_words += " ".join(tokens)+" "

wordcloud = WordCloud(width = 800, height = 800,
				background_color ='white',
				stopwords = stopwords,
				min_font_size = 10).generate(comment_words)

# plot the WordCloud image					 
plt.figure(figsize = (8, 8), facecolor = None)
plt.imshow(wordcloud)
plt.axis("off")
plt.tight_layout(pad = 0)

plt.show()


In [ ]:
# pip install bertopic


In [ ]:
import pandas as pd
import numpy as np
import gensim
from gensim import corpora
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from bertopic import BERTopic
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)  # Suppress warnings for a cleaner notebook output

# Load your data (assuming the descriptions are in a column named 'description')

# Possible Solutions to Speed Up
# 1. Reduce Dataset Size for Testing
descriptions_sample = cleaned_docs[:1000]  # Uncomment to use the first 1000 descriptions for testing

# 2. Lower the Number of Topics
num_topics = 5  # Adjust this value to a lower number for faster testing

# 3. Reduce Number of Passes in Gensim LDA
passes = 5  # Adjust this value for faster training

# 4. Optimize Vectorization by limiting max features
max_features = 5000  # Limit features to speed up vectorization

# 5. Parallel Processing for Gensim LDA
# You can use the LdaMulticore model for faster processing on multiple cores

# data = pd.read_csv('homeless_encampments.csv')
# descriptions = data['description'].dropna().tolist()


# Create dictionary and corpus for Gensim models
dictionary = corpora.Dictionary([doc.split() for doc in cleaned_docs])
corpus = [dictionary.doc2bow(doc.split()) for doc in cleaned_docs]

# Baseline Model 1: Latent Dirichlet Allocation (LDA) with Gensim
def lda_gensim_baseline(corpus, dictionary, num_topics=num_topics, passes=passes):
    lda_model = gensim.models.LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=num_topics,
        random_state=42,
        passes=passes,
        alpha='auto'
    )
    return lda_model

# Baseline Model 2: Latent Dirichlet Allocation (LDA) with Scikit-Learn
def lda_sklearn_baseline(descriptions, num_topics=5):
    vectorizer = CountVectorizer(stop_words='english', max_features=max_features)
    data_vectorized = vectorizer.fit_transform(descriptions)
    lda = LatentDirichletAllocation(n_components=num_topics, random_state=42)
    lda.fit(data_vectorized)
    return lda, vectorizer

# Baseline Model 3: Non-negative Matrix Factorization (NMF)
def nmf_baseline(descriptions, num_topics=5):
    vectorizer = TfidfVectorizer(stop_words='english', max_features=max_features)
    data_vectorized = vectorizer.fit_transform(descriptions)
    nmf_model = NMF(n_components=num_topics, random_state=42)
    nmf_model.fit(data_vectorized)
    return nmf_model, vectorizer

# Baseline Model 4: BERTopic
def bertopic_baseline(descriptions):
    topic_model = BERTopic()
    topics, probs = topic_model.fit_transform(descriptions)
    return topic_model

# Example usage
lda_gensim_model = lda_gensim_baseline(corpus, dictionary)
lda_sklearn_model, sklearn_vectorizer = lda_sklearn_baseline(cleaned_docs)
nmf_model, nmf_vectorizer = nmf_baseline(cleaned_docs)
bertopic_model = bertopic_baseline(cleaned_docs)

# Print topics for LDA Gensim
for idx, topic in lda_gensim_model.print_topics(-1):
    print(f"Topic {idx}: {topic}")

# Display topics for Scikit-Learn LDA
def display_topics(model, feature_names, num_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        print(f"Topic {topic_idx}: ", " ".join([feature_names[i] for i in topic.argsort()[:-num_top_words - 1:-1]]))

print("\nTopics from Scikit-Learn LDA:")
display_topics(lda_sklearn_model, sklearn_vectorizer.get_feature_names_out())

# Display topics for NMF
print("\nTopics from NMF:")
display_topics(nmf_model, nmf_vectorizer.get_feature_names_out())

# Display topics for BERTopic
print("\nTopics from BERTopic:")
print(bertopic_model.get_topic_info())

In [ ]:
#pip install top2vec

In [ ]:
from top2vec import Top2Vec

In [ ]:
#pip install top2vec\[sentence_encoders\]


In [ ]:
#pip install tensorflow tensorflow_hub tensorflow_text

In [ ]:
model = Top2Vec(cleaned_docs, ngram_vocab=True, workers=4, min_count=5)

In [ ]:
topic_sizes, topic_nums = model.get_topic_sizes()
print(topic_sizes)

In [ ]:
print(topic_nums)

## only showing two topics were formed

In [ ]:
topic_words, word_scores, topic_nums = model.get_topics()


In [ ]:
for words, scores, num in zip(topic_words, word_scores, topic_nums):
    print(num)
    print(f"Words: {words}")

## even with changing different hyperparameters, i can only get two topics, need to find an algorithm that can work better with this data. or a way to predefine topics but maybe that leads to bias in the modeling. either way, back to drawing board

In [ ]:
import re
import pandas as pd
import gensim
from gensim import corpora
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.models import LdaModel
import nltk
import os

# Download NLTK data if not already downloaded
nltk_data_dir = os.path.expanduser('~/nltk_data')
if not os.path.exists(nltk_data_dir):
    nltk.download('punkt', download_dir=nltk_data_dir)
    nltk.download('stopwords', download_dir=nltk_data_dir)
else:
    nltk.data.path.append(nltk_data_dir)

# Sample cleaned docs
docs = ['Active camp 4 days.', 'homeless man sleeping sidewalk apartment building, usually 6th Ave University Ave camping area.', 'Homeless encampment, near property fence provided address.', 'Bulky items block sidewalk abandoned occupied camps int park large items housed fence near basketball courts day. cite items and/or clean up. Trash reycle cans overflo', 'Encampment materials, clothing, bags, tent, etc. beach Devon Court Oceanfront Walk -- including bicycle.', 'Camp triangle sunset cliffs Nimitz', 'Homeless encampment', 'Homeless encampment weeks. Getting worse trash. camp backflow/ FDC 301 University Ave. 92103 faces 3rd ave Walgreens.', 'Homeless encampments park day', 'Homeless encampment. apartment building utility box address. HOA contacted.']

# Preprocess the text
def preprocess_text(doc):
    doc = doc.lower() # Convert to lowercase
    doc = re.sub(r'[^a-zA-Z0-9\s]', '', doc) # Remove special characters
    tokens = word_tokenize(doc) # Tokenize the text
    stop_words = set(stopwords.words('english')) # Define stopwords
    stop_words.update(['camp', 'homeless', 'encampment']) # Add domain-specific stopwords
    tokens = [token for token in tokens if token not in stop_words] # Remove stopwords
    return tokens

# Preprocess all docs
processed_docs = [preprocess_text(doc) for doc in docs]

# Create a dictionary and corpus for LDA
dictionary = corpora.Dictionary(processed_docs)
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

# Train the LDA model
num_topics = 3  # Set the number of topics
lda_model = LdaModel(corpus, num_topics=num_topics, id2word=dictionary, passes=15)

# Print the topics found by the model
topics = lda_model.print_topics(num_words=5)
for idx, topic in topics:
    print(f"Topic {idx}: {topic}")

# Example inference
test_doc = 'Homeless person setting up camp near park and using drugs.'
processed_test_doc = preprocess_text(test_doc)
bow_test_doc = dictionary.doc2bow(processed_test_doc)
topic_distribution = lda_model.get_document_topics(bow_test_doc)
print("\nTopic Distribution for the given document:")
print(topic_distribution)